# 0. Environment Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 0.1 Clone Repository & Install Dependencies

In [2]:
REPO_URL = "https://github.com/11erlangga/legal-rag-slm.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 78 (delta 41), reused 64 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 362.31 KiB | 6.47 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [3]:
import re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.0
    Uninstalling datasets-5.0.0:
      Successfully uninstalled datasets-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 81.1 MB/

In [4]:
!pip install -q rouge-score langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 0.2 Import Libraries & Secrets

In [5]:
import sys
sys.path.append(REPO_DIR)

from unsloth import FastLanguageModel
import torch
from trl import GRPOConfig, GRPOTrainer
import wandb

from src.data_utils import build_coldstart_dataset, load_split_dataset
from src.model_utils import run_coldstart_sft
from src.reward_functions import (
    format_reward_func,
    reasoning_length_reward_func,
    correctness_reward_func,
    language_reward_func,
    ROUGE_SIMILARITY_THRESHOLD,
)

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
WANDB_TOKEN = user_secrets.get_secret("WANDB_TOKEN")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Detected: {gpu_stats.name}. Max Memory = {max_memory} GB.")
else:
    print("WARNING: GPU Not Detected.")

GPU Detected: Tesla T4. Max Memory = 14.562 GB.


# 1. Experiment Tracking (WandB)

In [7]:
EXPERIMENT_NAME = "grpo-qwen25-3b"
USE_WANDB = True

if USE_WANDB:
    os.environ["WANDB_API_KEY"] = WANDB_TOKEN
    wandb.init(project="legal-rag-slm-grpo", name=EXPERIMENT_NAME)
    report_target = "wandb"
else:
    os.environ["WANDB_DISABLED"] = "true"
    report_target = "none"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kumo11 (kumo11_personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260913_073200-llgo5na9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run grpo-qwen25-3b
wandb: ⭐️ View project at https://wandb.ai/kumo11_personal/legal-rag-slm-grpo
wandb: 🚀 View run at https://wandb.ai/kumo11_personal/legal-rag-slm-grpo/runs/llgo5na9
wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


# 2. Setup & Load Model SFT

In [8]:
SFT_MODEL_REPO_ID = "11erlangga/sft-qwen25-3b-run1"

max_seq_length = 2048
max_prompt_length = 256

GRPO_LORA_R = 16
GRPO_LORA_ALPHA = 16

In [9]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_MODEL_REPO_ID,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    fast_inference = False, # vLLM untuk generate cepat saat GRPO rollout
    max_lora_rank = GRPO_LORA_R,
    gpu_memory_utilization = 0.6, # T4 14.7GB; turunkan lagi (0.5/0.4) kalau OOM saat load
)

# Fail-fast: kalau chat_template kosong, kemungkinan besar SFT_MODEL_REPO_ID
# salah (misal ke-resolve ke base model, bukan hasil push_to_hub_merged sendiri).
# Sama pola-nya dengan validasi di src/rag/generation.py::load_finetuned_model.
assert tokenizer.chat_template is not None, (
    "tokenizer.chat_template kosong — cek lagi SFT_MODEL_REPO_ID, "
    "kemungkinan bukan hasil push_to_hub_merged sendiri."
)

==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r = GRPO_LORA_R,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = GRPO_LORA_ALPHA,
    use_gradient_checkpointing = "unsloth",
    random_state = 1010,
)

Unsloth 2026.9.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


# 3. Sanity-Check 4 Reward Function

In [11]:
print(f"ROUGE_SIMILARITY_THRESHOLD = {ROUGE_SIMILARITY_THRESHOLD}")

ROUGE_SIMILARITY_THRESHOLD = 0.2


In [12]:
# TRL memanggil reward_func(prompts, completions, **kwargs) dan mengharapkan list[float].
# `completions` berbentuk list[list[{"role":..., "content":...}]] (konsisten dengan chat format).
# prompts ikut dilempar di sini juga supaya call signature match dengan yang TRL kirim saat training.

dummy_prompts = ["Apakah staf admin berhak dapat uang lembur?"]

dummy_completions_good = [[{
    "content": (
        "<think>\n"
        "Berdasarkan PP No. 35 Tahun 2021, pekerja yang lembur berhak atas upah lembur "
        "terlepas dari jabatannya, termasuk staf admin.\n"
        "</think>\n"
        "Ya, staf admin berhak mendapat uang lembur sesuai PP No. 35 Tahun 2021."
    )
}]]

dummy_completions_bad_no_tag = [[{"content": "Ya, staf admin berhak dapat uang lembur."}]]

dummy_completions_bad_hallucinated = [[{
    "content": "<think>a</think><think>b</think>\nJawaban singkat saja."
}]]

dummy_ground_truth = ["Ya, berhak, sesuai PP No. 35 Tahun 2021 tentang Waktu Kerja dan Waktu Istirahat."]

In [13]:
for name, comp in [
    ("good", dummy_completions_good),
    ("no_tag", dummy_completions_bad_no_tag),
    ("hallucinated_tags", dummy_completions_bad_hallucinated),
]:
    print(f"{name}")
    print("format_reward_func      :", format_reward_func(prompts=dummy_prompts, completions=comp))
    print("reasoning_length_reward :", reasoning_length_reward_func(prompts=dummy_prompts, completions=comp))
    print("correctness_reward_func :", correctness_reward_func(prompts=dummy_prompts, completions=comp, output=dummy_ground_truth))
    print("language_reward_func    :", language_reward_func(prompts=dummy_prompts, completions=comp))
    print()

good
format_reward_func      : [1.0]
reasoning_length_reward : [0.5]
correctness_reward_func : [1.0]
language_reward_func    : [1.0]

no_tag
format_reward_func      : [0.0]
reasoning_length_reward : [0.0]
correctness_reward_func : [0.0]
language_reward_func    : [1.0]

hallucinated_tags
format_reward_func      : [0.0]
reasoning_length_reward : [0.2]
correctness_reward_func : [0.0]
language_reward_func    : [1.0]



# 4. Prompt Formatting Khusus GRPO

In [14]:
   GRPO_SYSTEM_PROMPT = (
       "Kamu adalah asisten AI yang menjawab pertanyaan pengguna dengan akurat "
       "dan jelas dalam Bahasa Indonesia.\n"
       "WAJIB gunakan format berikut untuk SETIAP jawaban:\n"
       "<think>\n(tulis penalaranmu di sini, minimal beberapa kalimat)\n</think>\n"
       "(tulis jawaban akhir yang ringkas dan jelas di sini, dalam Bahasa Indonesia)"
   )

train_dataset, eval_dataset = load_split_dataset()

README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

In [15]:
def to_grpo_prompt(example):
    return {
        "prompt": [
            {"role": "system", "content": GRPO_SYSTEM_PROMPT},
            {"role": "user", "content": example["input"]},
        ],
        # dipertahankan sebagai kolom terpisah untuk correctness_reward_func (ground truth)
        "output": example["output"],
    }

grpo_train_dataset = train_dataset.map(to_grpo_prompt, remove_columns=train_dataset.column_names)
grpo_eval_dataset = eval_dataset.map(to_grpo_prompt, remove_columns=eval_dataset.column_names)

print(grpo_train_dataset[0])

Map:   0%|          | 0/47470 [00:00<?, ? examples/s]

Map:   0%|          | 0/2499 [00:00<?, ? examples/s]

{'output': 'Berikut adalah desain dan penjelasan untuk antarmuka pengguna dari aplikasi manajemen tugas berbasis AI:\n\n1. Halaman Utama: Saat membuka aplikasi, pengguna akan disambut dengan halaman utama yang bersih dan terorganisir. Halaman ini menampilkan ringkasan tugas yang masih harus dikerjakan, kategori tugas, dan kotak pencarian. Avatar asisten virtual dapat hadir di pojok layar untuk memberikan bantuan dan menjalankan perintah suara.\n\n2. Pembuatan Tugas: Pengguna dapat dengan cepat menambahkan tugas dengan mengklik tombol "tambah tugas", yang akan membuka formulir di mana mereka dapat mengisi deskripsi tugas, tipe (pekerjaan, pribadi, dll.), tanggal jatuh tempo, tingkat prioritas, dan lainnya. AI dapat memberikan saran berdasarkan kata kunci dan perilaku masa lalu, misalnya, \'Apakah Anda ingin menambahkan ini sebagai tugas berulang setiap hari Senin?\'.\n\n3. Daftar Tugas: Daftar tugas menampilkan semua tugas secara kronologis, dan dapat disaring berdasarkan kategori, prio

# 5. Inject Think-Tag Format via Cold-Start SFT

In [16]:
train_dataset, _ = load_split_dataset(test_size=0.05, seed=1010)

coldstart_dataset = build_coldstart_dataset(
    train_dataset,
    tokenizer,
    system_prompt=GRPO_SYSTEM_PROMPT,
    n_samples=300,
    seed=1010,
)

print(coldstart_dataset[0]["text"])

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

<|im_start|>system
Kamu adalah asisten AI yang menjawab pertanyaan pengguna dengan akurat dan jelas dalam Bahasa Indonesia.
WAJIB gunakan format berikut untuk SETIAP jawaban:
<think>
(tulis penalaranmu di sini, minimal beberapa kalimat)
</think>
(tulis jawaban akhir yang ringkas dan jelas di sini, dalam Bahasa Indonesia)<|im_end|>
<|im_start|>user
Membuat daftar langkah-langkah untuk belajar bahasa baru.
<|im_end|>
<|im_start|>assistant
<think>
Saya perlu mempertimbangkan membuat daftar langkah-langkah untuk belajar bahasa sebelum memberikan jawaban akhir.
</think>
Berikut adalah beberapa langkah yang dapat membimbing Anda saat memulai perjalanan belajar bahasa baru:

1. Tentukan motivasi Anda: Mulailah dengan mengidentifikasi alasan mengapa Anda ingin belajar bahasa tersebut. Apakah Anda memerlukannya untuk pekerjaan atau perjalanan? Apakah Anda belajar untuk berkomunikasi dengan teman atau keluarga? Ini akan membantu Anda tetap termotivasi dan fokus saat belajar.

2. Tetapkan tujuan 

In [17]:
coldstart_trainer = run_coldstart_sft(
    model,
    tokenizer,
    coldstart_dataset,
    output_dir="coldstart_checkpoint",
    max_steps=150,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 4 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.543000
20,1.186800
30,0.914600
40,0.910800
50,0.833600
60,0.838000
70,0.794200
80,0.753100
90,0.773100
100,0.795400


In [18]:
# Sanity Check
def generate_manual_completions(model, tokenizer, eval_dataset, n_samples=8, max_new_tokens=400):
    FastLanguageModel.for_inference(model)
    
    results = []
    samples = eval_dataset.select(range(n_samples))
    
    for i, ex in enumerate(samples):
        messages = ex["prompt"]  # [{"role": "system", ...}, {"role": "user", ...}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
        )
        
        raw_completion_only = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False
        )
        
        results.append({
            "idx": i,
            "prompt": messages[-1]["content"],
            "raw_completion_repr": repr(raw_completion_only),
            "has_think_open": "<think>" in raw_completion_only,
            "has_think_close": "</think>" in raw_completion_only,
        })
        
        print(f"Sample {i}")
        print(repr(raw_completion_only))
        print(f"has <think>: {results[-1]['has_think_open']} | has </think>: {results[-1]['has_think_close']}")

    return results

diagnosis_after_coldstart = generate_manual_completions(
    model, tokenizer, grpo_eval_dataset, n_samples=3
)

Sample 0
'<think>\nUntuk menjawab ini, saya perlu memahami inti permintaan terkait buatlah kalimat. Berikut jawabannya.\n</think>\n"Saya membaca artikel tersebut dengan hati-hati, memastikan bahwa saya tidak melewatkan apa pun."<|im_end|>'
has <think>: True | has </think>: True
Sample 1
'<think>\nPermintaan ini berkaitan dengan tuliskan ulang kalimat berikut. Saya akan memberikan jawaban yang relevan dan ringkas.\n</think>\nRohan dan Abhisek akan pergi ke pasar. Mereka akan pergi bersama-sama.<|im_end|>'
has <think>: True | has </think>: True
Sample 2
'<think>\nPermintaan ini berkaitan dengan berdasarkan pernyataan yang diberikan, buatlah. Saya akan memberikan jawaban yang relevan dan ringkas.\n</think>\nBerdasarkan pernyataan "Tim membutuhkan lebih banyak latihan," kesimpulan yang dapat diambil adalah bahwa tim memerlukan peningkatan pada latihan mereka.<|im_end|>'
has <think>: True | has </think>: True


# 6. Configure GRPO Training

> Parameter paling kritis untuk OOM mitigation di T4: `num_generations` dan `max_completion_length`.

In [19]:
FastLanguageModel.for_training(model)

TRIAL_RUN = False  # set False setelah trial run aman dari OOM

training_args = GRPOConfig(
    use_vllm = False,
    learning_rate = 5e-6, # lebih kecil dari SFT —> hindari menjauh terlalu jauh dari model SFT
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,
    num_generations = 4 if TRIAL_RUN else 8, # naikkan setelah trial run terbukti tidak OOM
    max_prompt_length = max_prompt_length,
    max_completion_length = 300 if TRIAL_RUN else 400,
    beta = 0.04, # KL penalty coefficient —> default TRL, tuning kalau reward-chasing
    max_steps = 75 if TRIAL_RUN else 350,
    save_steps = 75 if TRIAL_RUN else 50,
    max_grad_norm = 0.1,
    report_to = "wandb" if USE_WANDB else False,
    run_name = "grpo-qwen25-3b-trial" if TRIAL_RUN else "grpo-qwen25-3b-full",
    output_dir = "outputs",
    seed = 1010,
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


# 7. Train Model with GRPO

In [20]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward_func,
        reasoning_length_reward_func,
        correctness_reward_func,
        language_reward_func,
    ],
    args = training_args,
    train_dataset = grpo_train_dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 47,470 | Num Epochs = 1 | Total steps = 350
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 32768, 'temperature': 0.7, 'top_p': 0.8, 'repetition_penalty': 1.05}. If this is not desired, please set these values explicitly.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / reasoning_length_reward_func / mean,rewards / reasoning_length_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std,rewards / language_reward_func / mean,rewards / language_reward_func / std
1,0.033800,3.000000,0.534522,62.875000,50.000000,69.000000,0.000000,62.875000,50.000000,69.000000,0.844533,1.000000,0.000000,0.500000,0.000000,0.500000,0.534522,1.000000,0.000000
2,0.007500,2.500000,0.000000,213.375000,147.000000,326.000000,0.000000,213.375000,147.000000,326.000000,0.186636,1.000000,0.000000,0.500000,0.000000,0.000000,0.000000,1.000000,0.000000
3,0.010500,3.500000,0.000000,236.375000,197.000000,275.000000,0.000000,236.375000,197.000000,275.000000,0.262866,1.000000,0.000000,0.500000,0.000000,1.000000,0.000000,1.000000,0.000000
4,0.017400,2.750000,0.462910,121.625000,82.000000,183.000000,0.000000,121.625000,82.000000,183.000000,0.433794,1.000000,0.000000,0.500000,0.000000,0.250000,0.462910,1.000000,0.000000
5,0.006600,2.500000,0.000000,400.000000,400.000000,400.000000,1.000000,0.000000,0.000000,0.000000,0.164477,1.000000,0.000000,0.500000,0.000000,0.000000,0.000000,1.000000,0.000000
6,0.006200,2.500000,0.000000,338.500000,195.000000,400.000000,0.625000,236.000000,195.000000,276.000000,0.155203,1.000000,0.000000,0.500000,0.000000,0.000000,0.000000,1.000000,0.000000
7,0.008300,3.125000,0.517549,234.000000,155.000000,300.000000,0.000000,234.000000,155.000000,300.000000,0.208499,1.000000,0.000000,0.500000,0.000000,0.625000,0.517549,1.000000,0.000000
8,0.028700,2.500000,0.000000,75.875000,54.000000,102.000000,0.000000,75.875000,54.000000,102.000000,0.717665,1.000000,0.000000,0.500000,0.000000,0.000000,0.000000,1.000000,0.000000
9,0.015100,2.750000,0.462910,173.875000,149.000000,193.000000,0.000000,173.875000,149.000000,193.000000,0.378059,1.000000,0.000000,0.500000,0.000000,0.250000,0.462910,1.000000,0.000000
10,0.008400,3.250000,0.462910,173.250000,96.000000,324.000000,0.000000,173.250000,96.000000,324.000000,0.211116,1.000000,0.000000,0.500000,0.000000,0.750000,0.462910,1.000000,0.000000


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


TrainOutput(global_step=350, training_loss=0.014570429962727107, metrics={'train_runtime': 14433.3914, 'train_samples_per_second': 0.194, 'train_steps_per_second': 0.024, 'total_flos': 0.0, 'train_loss': 0.014570429962727107})

In [21]:
if USE_WANDB:
    wandb.finish()

wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml; uploading output.log
wandb: 
wandb: Run history:
wandb:           profiling/Time taken: UnslothGRPOTrainer._calculate_rewards ▁▂▂▁█▃▅▄▁▂▇▂▇▂▂▃▁▂▃▁▆▁▂▄▂▅▂▂▃▆▄▁█▂▆▂▂▂█▁
wandb:              profiling/Time taken: UnslothGRPOTrainer._prepare_inputs ▂▆▇▇▆▇▂█▂▇▅▇▇▂▃▂▂▂▁▁▇▃▇▇▁▇▁▁▆▅▇▇▂▇▇▁▇▇▅▇
wandb:      profiling/Time taken: UnslothGRPOTrainer.correctness_reward_func ▁▁▃█▃▁▁▂▂▂▁▄▁▁▃▄▁▁▁▁▁▁▁▁▁▂▁▁▆▁▂▄▃▁▃▁▅▅▂▁
wandb:           profiling/Time taken: UnslothGRPOTrainer.format_reward_func ▂▃▄▂▇▁▅▃▃▄▃▅▅▁▇▁██▁▂▂▃▂▂▄▂▃▄▄▄▅▄▆▂▂▁▁▆▂▂
wandb:         profiling/Time taken: UnslothGRPOTrainer.language_reward_func ▃▂▃▂▂▁▃▂▃▂▂▁▃▂▁▄▃▃█▃▄▁▁▁▁▃▂▃▂▃▃▂▂▂▃▂▃▂▂▄
wandb: profiling/Time taken: UnslothGRPOTrainer.reasoning_length_reward_func ▂▃▆█▂▂▄▄▄▆▅▆▅▅▅▅▄▃▁▂▄▃▃▂▅▂▄▃▁▅▃▃▁▂▅▂▃▆▄▃
wandb:        profiling/Time taken: UnslothGRPOTrainer.transformers.generate ▆▆█▂▂██████▁▄▃██▄█▂▂▂██▁█▂▆▁▃▁▇█▂▁▂▄███▂
wandb:                              

# 8. Push Model to HuggingFace Hub

In [22]:
HF_REPO_ID = f"11erlangga/{EXPERIMENT_NAME}"

model.push_to_hub_merged(
    HF_REPO_ID,
    tokenizer,
    save_method = "merged_16bit",
    token = HF_TOKEN,
)

print(f"GRPO: https://huggingface.co/{HF_REPO_ID}")

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:18<01:18, 78.53s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:02<00:00, 61.42s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/11erlangga/grpo-qwen25-3b`
GRPO: https://huggingface.co/11erlangga/grpo-qwen25-3b


# 9. Inference Sanity Check

In [23]:
FastLanguageModel.for_inference(model)

test_prompt = tokenizer.apply_chat_template([
    {"role": "system", "content": GRPO_SYSTEM_PROMPT},
    {"role": "user", "content": (
        "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. "
        "Apakah saya berhak dapat uang lembur?"
    )},
], tokenize=False, add_generation_prompt=True)

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=400,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)

output_text = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)
print(output_text)

<think>
Pertanyaan ini meminta saya untuk sesuaikan jawaban berdasarkan saran terkait isu konteks. Berikut jawabannya.
</think>
Tentukan apakah Anda berhak mendapatkan gaji tambahan tergantung pada aturan perusahaan dan kebijakan terkait lembur. Biasanya, jika Anda bekerja di atas jam kerja standar yang ditetapkan oleh perusahaan, Anda mungkin memiliki hak untuk gaji tambahan. Namun, penting untuk memeriksa aturan dan keteraturan perusahaan Anda secara langsung karena mereka dapat berbeda dari satu tempat kerja ke tempat lainnya.
